In [2]:
import os
import re
import subprocess
import pandas as pd
from collections import defaultdict

SLURM_LOG_DIR = "/n/scratch/users/a/adm808/slurm_logs"
OUT_PREFIX = "genes_"
CELL_TYPES = ["Ast", "Ex", "In", "Mic", "Oli", "Opc"]

def parse_cell_type(out_path):
    try:
        with open(out_path, "r", errors="ignore") as f:
            first_line = f.readline().strip()

        # Expected pattern:
        # "Running leave one out experiment with experiment maximal and sample: Mic and 1"
        match = re.search(r"sample:\s*(Ast|Ex|In|Mic|Oli|Opc)\b", first_line)
        if match:
            return match.group(1)

    except Exception:
        pass

    return "Unknown"

def sacct_job(jobid):
    """Query sacct for elapsed time and MaxRSS"""
    cmd = [
        "sacct",
        "-j", jobid,
        "--format=JobID,Elapsed,MaxRSS,State",
        "--units=G",
        "--noheader"
    ]
    try:
        out = subprocess.check_output(cmd, text=True)
    except subprocess.CalledProcessError:
        return None

    elapsed = None
    maxrss = None

    for line in out.splitlines():
        if ".ba+" in line:
            parts = line.split()
            elapsed = parts[1]
            maxrss = parts[2]

    return elapsed, maxrss

def time_to_hours(t):
    """HH:MM:SS → hours"""
    h, m, s = t.split(":")
    return int(h) + int(m)/60 + int(s)/3600

records = []

for fname in os.listdir(SLURM_LOG_DIR):
    if not fname.startswith(OUT_PREFIX) or not fname.endswith(".out"):
        continue

    jobid = fname.replace(OUT_PREFIX, "").replace(".out", "")
    out_path = os.path.join(SLURM_LOG_DIR, fname)

    cell_type = parse_cell_type(out_path)
    sacct_res = sacct_job(jobid)

    if sacct_res is None:
        continue

    elapsed, maxrss = sacct_res
    if elapsed is None or maxrss is None:
        continue

    records.append({
        "JobID": jobid,
        "CellType": cell_type,
        "Runtime_hours": time_to_hours(elapsed),
        "PeakMemory_GB": float(maxrss.replace("G", ""))
    })

df = pd.DataFrame(records)

# Aggregate per cell type
summary = (
    df.groupby("CellType")
      .agg(
          n_jobs=("JobID", "count"),
          mean_runtime_hours=("Runtime_hours", "mean"),
          max_runtime_hours=("Runtime_hours", "max"),
          mean_peak_memory_GB=("PeakMemory_GB", "mean"),
          max_peak_memory_GB=("PeakMemory_GB", "max")
      )
      .reset_index()
)

# Write Excel
out_xlsx = "TriSCOPE_runtime_summary.xlsx"
with pd.ExcelWriter(out_xlsx, engine="xlsxwriter") as writer:
    df.to_excel(writer, sheet_name="RawJobs", index=False)
    summary.to_excel(writer, sheet_name="PerCellTypeSummary", index=False)

print(f"Saved runtime summary to {out_xlsx}")

Saved runtime summary to TriSCOPE_runtime_summary.xlsx


In [3]:
import os
import re
import subprocess
import pandas as pd

SLURM_LOG_DIR = "/n/scratch/users/a/adm808/slurm_logs"
OUT_PREFIX = "genes_"

def parse_cell_type(out_path):
    try:
        with open(out_path, "r", errors="ignore") as f:
            first_line = f.readline().strip()

        match = re.search(r"sample:\s*(Ast|Ex|In|Mic|Oli|Opc)\b", first_line)
        if match:
            return match.group(1)
    except Exception:
        pass
    return "Unknown"

def sacct_job(jobid):
    cmd = [
        "sacct",
        "-j", jobid,
        "--format=JobID,Elapsed,MaxRSS,State",
        "--units=G",
        "--noheader"
    ]
    try:
        out = subprocess.check_output(cmd, text=True)
    except subprocess.CalledProcessError:
        return None

    elapsed = None
    maxrss = None
    for line in out.splitlines():
        if ".ba+" in line:
            parts = line.split()
            elapsed = parts[1]
            maxrss = parts[2]
    return elapsed, maxrss

def time_to_hours(t):
    h, m, s = t.split(":")
    return int(h) + int(m)/60 + int(s)/3600

records = []

for fname in os.listdir(SLURM_LOG_DIR):
    if not fname.startswith(OUT_PREFIX) or not fname.endswith(".out"):
        continue

    jobid = fname.replace(OUT_PREFIX, "").replace(".out", "")
    out_path = os.path.join(SLURM_LOG_DIR, fname)

    cell_type = parse_cell_type(out_path)
    sacct_res = sacct_job(jobid)
    if sacct_res is None:
        continue

    elapsed, maxrss = sacct_res
    if elapsed is None or maxrss is None:
        continue

    records.append({
        "JobID": jobid,
        "CellType": cell_type,
        "Runtime_hours": time_to_hours(elapsed),
        "PeakMemory_GB": float(maxrss.replace("G", ""))
    })

df = pd.DataFrame(records)

# -------------------------
# PRINT RAW JOBS
# -------------------------
print("\n=== Raw predictive modeling jobs ===\n")
print(df.sort_values(["CellType", "Runtime_hours"]).to_string(index=False))

# -------------------------
# PRINT PER CELL TYPE SUMMARY
# -------------------------
summary = (
    df.groupby("CellType")
      .agg(
          n_jobs=("JobID", "count"),
          mean_runtime_hours=("Runtime_hours", "mean"),
          max_runtime_hours=("Runtime_hours", "max"),
          mean_peak_memory_GB=("PeakMemory_GB", "mean"),
          max_peak_memory_GB=("PeakMemory_GB", "max")
      )
      .reset_index()
)

print("\n=== Per cell-type runtime & memory summary ===\n")
print(summary.to_string(index=False))

# -------------------------
# COPY-PASTE FRIENDLY OUTPUT
# -------------------------
print("\n=== Spreadsheet-ready summary ===\n")
for _, row in summary.iterrows():
    print(
        f"{row['CellType']}: "
        f"{row['max_runtime_hours']:.2f} h, "
        f"{row['max_peak_memory_GB']:.1f} GB "
        f"(n={int(row['n_jobs'])})"
    )


=== Raw predictive modeling jobs ===

   JobID CellType  Runtime_hours  PeakMemory_GB
23917880      Ast       0.778333          13.61
23915839      Ast       5.383333          13.92
23914947      Ast       5.491389          10.98
23914955      Ast       5.505278          11.06
23915836      Ast       5.511111          13.19
23914990      Ast       5.521944          17.62
23914942      Ast       5.532222          13.67
23917883      Ast       5.534444          11.98
23914972      Ast       5.540000          19.56
23917879      Ast       5.573056          10.91
23915834      Ast       5.615278          14.60
23915840      Ast       5.624722          13.92
23917884      Ast       5.650278          11.50
23915837      Ast       5.773333          14.54
23917881      Ast       5.775556          13.61
23915379      Ast       5.776667          13.67
23915387      Ast       5.789444          14.00
23915384      Ast       5.794167          13.29
23915381      Ast       5.795556          13.68
2

In [9]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import re
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import combine_pvalues, spearmanr
from statsmodels.stats.multitest import multipletests
from statsmodels.nonparametric.smoothers_lowess import lowess
import seaborn as sns
import shutil
import time
import resource
START_TIME = time.time()

# --- Set up Paths ---
expr_path = "/n/groups/patel/adithya/CellMatrix_with_genenames.parquet"
meta_path = "/n/groups/patel/adithya/New_CellMetadataSyn1848517.parquet"
output_path = "/n/scratch/users/a/adm808/runtime_test_scraps"
os.makedirs(output_path, exist_ok=True)

# --- Load Data ---
meta_df = pd.read_parquet(meta_path)
meta_df['AD'] = meta_df['dcfdx_lv'].isin([4.0, 5.0]).map({True: 'AD', False: 'Control'}).astype('category')
print("AD status defined based on 'dcfdx':")
print(meta_df['AD'].value_counts())

expr_df = pd.read_parquet(expr_path)
expr_df.set_index('index', inplace=True)
expr_df_transposed = expr_df.T

# --- Analysis Settings ---
all_subclusters = meta_df['Subcluster'].unique()
# all_subclusters = ['Ex6']  # Uncomment to test single subcluster

random_seeds = [41, 42, 43]
# 57, 62, 72, 73

validation_results = []
summary_roc_data = []
summary_roc_data_cerad = []
roc_plot_data_cerad = []

# --- Main Loop ---
for current_subcluster in all_subclusters:
    print(f"\n--- Processing Subcluster: {current_subcluster} ---")

    meta_sub = meta_df[meta_df['Subcluster'] == current_subcluster]

    if meta_sub.shape[0] < 50 or len(meta_sub['AD'].unique()) < 2:
        print(f"Skipping {current_subcluster}: <50 cells or only one AD/Control group.")
        continue

    safe_subcluster_name = re.sub(r'[^a-zA-Z0-9_.-]', '_', current_subcluster)
    subcluster_cells = meta_sub['TAG'].tolist()
    expr_sub = expr_df_transposed.loc[subcluster_cells]

    # --- Create AnnData ---
    adata_template = sc.AnnData(X=expr_sub)
    adata_template.obs = meta_sub.set_index('TAG').loc[adata_template.obs.index]

    # --- NON-CIRCULAR: Use HVGs instead of DEGs ---
    print("Selecting HVGs for pseudotime (non-circular approach)...")
    sc.pp.normalize_total(adata_template, target_sum=1e4)
    sc.pp.log1p(adata_template)
    
    # Select top 2000 HVGs (or all genes if fewer available)
    n_top = min(2000, adata_template.n_vars - 1)
    sc.pp.highly_variable_genes(adata_template, n_top_genes=n_top, flavor='seurat')
    hvg_genes = adata_template.var[adata_template.var['highly_variable']].index.tolist()

    if len(hvg_genes) < 50:
        print(f"Skipping {current_subcluster}: only {len(hvg_genes)} HVGs found.")
        continue

    adata_template = adata_template[:, hvg_genes].copy()
    print(f"Using {len(hvg_genes)} HVGs to build trajectories.")

    # --- PCA (no Harmony - not needed within subclusters) ---
    n_pcs = min(30, len(hvg_genes) - 1)
    sc.pp.pca(adata_template, n_comps=n_pcs)
    print(f"PCA complete: {adata_template.obsm['X_pca'].shape}")

    # --- FAST neighbor graph ---
    print(f"Building neighbor graph for {adata_template.n_obs} cells...")
    sc.pp.neighbors(adata_template, n_neighbors=10, n_pcs=n_pcs, method='umap')
    print("Neighbor graph complete!")

    # --- Downstream: UMAP, clustering, diffusion map ---
    sc.tl.umap(adata_template)
    sc.tl.leiden(adata_template, resolution=0.5)
    sc.tl.diffmap(adata_template)

    # --- Loop over random seeds to test trajectory robustness ---
    subcluster_aucs = []
    roc_plot_data = []
    top_trajectory_genes_from_gt = []  # will be filled after seed loop

    subcluster_pvals, subcluster_odds_ratios = [], []
    all_fprs, all_tprs = [], []  # For mean ROC over seeds

    cerad_aucs = []
    cerad_odds_ratios = []
    cerad_pvals = []
    cerad_or_cis = []

    # Store pseudotime per seed (for later averaging)
    pseudotime_per_seed = []

    for seed in random_seeds:
        print(f"\n  -- Running analysis for seed: {seed} --")
        
        # Create a fresh copy for each run
        adata = adata_template.copy()

        # --- Select Root Cell Based on ceradsc and cogdx (with current seed) ---
        subcluster_meta_dpt = adata.obs.copy()
        subcluster_meta_dpt['ceradsc'] = pd.to_numeric(subcluster_meta_dpt['ceradsc'], errors='coerce')
        subcluster_meta_dpt['cogdx'] = pd.to_numeric(subcluster_meta_dpt['cogdx'], errors='coerce')
        subcluster_meta_dpt.dropna(subset=['ceradsc', 'cogdx'], inplace=True)
        
        if subcluster_meta_dpt.empty:
            print("  Skipping trajectory ordering: No cells with valid ceradsc/cogdx values.")
            break

        max_ceradsc = subcluster_meta_dpt['ceradsc'].max()
        root_candidates = subcluster_meta_dpt[subcluster_meta_dpt['ceradsc'] == max_ceradsc]
        min_cogdx = root_candidates['cogdx'].min()
        root_candidates = root_candidates[root_candidates['cogdx'] == min_cogdx]
        
        root_cell_id = root_candidates.sample(n=1, random_state=seed).index[0]
        adata.uns['iroot'] = np.where(adata.obs_names == root_cell_id)[0][0]

        # --- Calculate Pseudotime (DPT) ---
        sc.tl.dpt(adata)

        # Store pseudotime for this seed (full vector, all cells)
        pseudotime_per_seed.append(adata.obs['dpt_pseudotime'].to_numpy())

        # --- Calculate AUC for the current seed ---
        validation_df = adata.obs[['dpt_pseudotime', 'AD']].copy()
        validation_df.dropna(subset=['dpt_pseudotime'], inplace=True)
        y_true = (validation_df['AD'] == 'AD').astype(int)
        y_score = validation_df['dpt_pseudotime']
        
        auc_score = roc_auc_score(y_true=y_true, y_score=y_score)
        subcluster_aucs.append(auc_score)
        print(f"  --> Seed {seed}: AUC = {auc_score:.4f}")

        # Logistic regression for this seed
        X = sm.add_constant(validation_df['dpt_pseudotime'])
        if len(y_true.unique()) < 2:
            print(f"  Skipping seed {seed}: only one class present after dropping NaNs.")
            continue
            
        log_reg = sm.Logit(y_true, X).fit(disp=0)
        subcluster_pvals.append(log_reg.pvalues['dpt_pseudotime'])
        subcluster_odds_ratios.append(np.exp(log_reg.params['dpt_pseudotime']))

        # ROC curve for this seed (clinical AD)
        fpr, tpr, _ = roc_curve(y_true=y_true, y_score=y_score)
        all_fprs.append(fpr)
        all_tprs.append(tpr)

        if auc_score > 0.55:
            roc_plot_data.append({'fpr': fpr, 'tpr': tpr, 'auc': auc_score, 'seed': seed})

        # CERAD-based classifier
        try:
            cerad_df = adata.obs[['dpt_pseudotime']].copy()
            cerad_df['TAG'] = cerad_df.index

            meta_subset = meta_df[['TAG', 'ceradsc']].dropna()
            meta_subset['ceradsc'] = pd.to_numeric(meta_subset['ceradsc'], errors='coerce')

            cerad_df = cerad_df.merge(meta_subset, on="TAG", how="left")
            cerad_df = cerad_df.dropna(subset=["ceradsc", "dpt_pseudotime"])
            cerad_df['AD_cerad'] = cerad_df['ceradsc'].isin([1, 2]).astype(int)

            if cerad_df['AD_cerad'].nunique() < 2:
                print(f"  CERAD classifier skipped for seed {seed}: only one class present.")
            else:
                X_cerad = sm.add_constant(cerad_df['dpt_pseudotime'])
                y_cerad = cerad_df['AD_cerad']
                cerad_model = sm.Logit(y_cerad, X_cerad).fit(disp=0)

                cerad_auc = roc_auc_score(y_cerad, cerad_df['dpt_pseudotime'])
                cerad_or = np.exp(cerad_model.params['dpt_pseudotime'])
                cerad_p = cerad_model.pvalues['dpt_pseudotime']
                cerad_conf = cerad_model.conf_int().loc['dpt_pseudotime']
                cerad_or_lower = np.exp(cerad_conf[0])
                cerad_or_upper = np.exp(cerad_conf[1])

                cerad_aucs.append(cerad_auc)
                cerad_odds_ratios.append(cerad_or)
                cerad_pvals.append(cerad_p)
                cerad_or_cis.append((cerad_or_lower, cerad_or_upper))

                fpr_cerad, tpr_cerad, _ = roc_curve(y_cerad, cerad_df['dpt_pseudotime'])

                if cerad_auc > 0.55:
                    roc_plot_data_cerad.append({
                        'fpr': fpr_cerad,
                        'tpr': tpr_cerad,
                        'auc': cerad_auc,
                        'seed': seed
                    })

                print(f"  CERAD AUC (seed {seed}): {cerad_auc:.3f}")
        except Exception as e:
            print(f"  CERAD model failed for seed {seed}: {e}")

    # --- After all seeds are run, aggregate results and create combined outputs ---
    if not subcluster_aucs:
        print(f"No valid results generated for {current_subcluster}. Skipping final aggregation.")
        continue

    # === 1) Aggregate pseudotime across seeds (Option A: rank-normalized) ===
    if pseudotime_per_seed:
        pseudo_array = np.column_stack(pseudotime_per_seed)  # cells x n_seeds_used
        n_cells, n_seeds_used = pseudo_array.shape
        ranked = np.zeros_like(pseudo_array, dtype=float)

        for j in range(n_seeds_used):
            col = pseudo_array[:, j]
            valid = ~np.isnan(col)
            ranks = np.empty_like(col, dtype=float)
            ranks[~valid] = np.nan

            # rank 0..(n_valid-1)
            valid_vals = col[valid]
            order = np.argsort(valid_vals)
            rank_values = np.empty_like(order, dtype=float)
            rank_values[order] = np.arange(len(order), dtype=float)

            ranks[valid] = rank_values
            denom = max(len(order) - 1, 1)
            ranked[:, j] = ranks / denom

        mean_pseudotime = np.nanmean(ranked, axis=1)
        adata_template.obs['mean_pseudotime'] = mean_pseudotime
        adata_template.obs['TAG'] = adata_template.obs.index

        # --- Save GAM input using mean pseudotime ---
        gam_df = adata_template.obs[['projid', 'AD', 'TAG']].copy()
        gam_df['AD_binary'] = (gam_df['AD'] == 'AD').astype(int)
        gam_df['dpt_pseudotime'] = mean_pseudotime
        gam_df = gam_df.dropna(subset=['projid', 'AD_binary', 'TAG', 'dpt_pseudotime'])
        gam_df.to_csv(os.path.join(output_path, f"pseudotime_gam_input_{safe_subcluster_name}.csv"), index=False)

        # === 2) PC selection by correlation with mean pseudotime ===
        pc_scores = adata_template.obsm['X_pca'][:, :n_pcs]  # cells x PCs
        pc_correlations = []
        for k in range(pc_scores.shape[1]):
            rho, _ = spearmanr(pc_scores[:, k], mean_pseudotime)
            pc_correlations.append(rho)
        best_pc = int(np.nanargmax(np.abs(pc_correlations)))

        # "Average" gene loadings across seeds (PCA is shared; average = original)
        loadings_matrix = adata_template.varm['PCs']  # genes x PCs
        avg_loadings = loadings_matrix[:, best_pc]

        # all_pc_genes_df.to_csv(os.path.join(output_path, f"all_pc_loadings_{safe_subcluster_name}.csv"), index=False)

        top_idx = np.argsort(np.abs(avg_loadings))[::-1][:10]
        top_trajectory_genes_from_gt = adata_template.var_names[top_idx].tolist()
        print(f"  Top trajectory-informing genes (PC{best_pc + 1}): {top_trajectory_genes_from_gt}")

        top_50 = np.argsort(np.abs(avg_loadings))[::-1][:50]
        top_50_trajectory_genes_from_gt = adata_template.var_names[top_50].tolist()

        # --- Save top 50 gene loadings in rank order ---
        top50_df = pd.DataFrame({
            'rank': np.arange(1, 51),
            'gene': adata_template.var_names[top_50],
            'loading': avg_loadings[top_50],
            'abs_loading': np.abs(avg_loadings[top_50])
        })

        # Save to subcluster-specific CSV
        top50_path = os.path.join(output_path, f"top50_PC_loadings_{safe_subcluster_name}.csv")
        top50_df.to_csv(top50_path, index=False)
        print(f"Saved top-50 PC loadings → {top50_path}")
    
        # === 3a) UMAP plot colored by mean pseudotime ===
        plot_filename_umap = os.path.join(output_path, f"umap_pseudotime_{safe_subcluster_name}.png")
        sc.pl.umap(
            adata_template,
            color='mean_pseudotime',
            title=f'{current_subcluster}: Mean Pseudotime Trajectory',
            show=False,
            save=f"_{safe_subcluster_name}_umap.png",
            cmap="viridis"
        )
        shutil.move(f'figures/umap_{safe_subcluster_name}_umap.png', plot_filename_umap)

        # === 3b) Smoothed expression of top genes vs mean pseudotime ===
        if top_trajectory_genes_from_gt:
            fig, ax = plt.subplots(figsize=(10, 6))
            genes_to_plot = top_trajectory_genes_from_gt[:4]
            colors = sns.color_palette("colorblind", n_colors=len(genes_to_plot))

            pseudo_vals = mean_pseudotime
            valid_idx = ~np.isnan(pseudo_vals)
            sorted_idx = np.argsort(pseudo_vals[valid_idx])

            for gene, color in zip(genes_to_plot, colors):
                expr_vals = adata_template[:, gene].X
                # Handle sparse
                if not isinstance(expr_vals, np.ndarray):
                    expr_vals = expr_vals.toarray()
                expr_vals = np.asarray(expr_vals).flatten()

                expr_valid = expr_vals[valid_idx][sorted_idx]
                pseudo_sorted = pseudo_vals[valid_idx][sorted_idx]

                smoothed = lowess(expr_valid, pseudo_sorted, frac=0.3)
                smoothed_z = (smoothed[:, 1] - smoothed[:, 1].mean()) / smoothed[:, 1].std()
                ax.plot(smoothed[:, 0], smoothed_z, label=gene, color=color)

            ax.set_xlabel('Mean Pseudotime (rank-normalized)')
            ax.set_ylabel('Z-scored Expression')
            ax.set_title(f'{current_subcluster}: Smoothed Expression of Top Genes')
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
            fig.savefig(os.path.join(output_path, f"lineplot_top_genes_{safe_subcluster_name}.png"),
                        bbox_inches='tight')
            plt.close(fig)

    # === Aggregate CERAD stats across seeds ===
    if cerad_aucs:
        mean_auc_cerad = np.mean(cerad_aucs)
        se_auc_cerad = np.std(cerad_aucs, ddof=1) / np.sqrt(len(cerad_aucs))
        auc_cerad_ci = (mean_auc_cerad - 1.96 * se_auc_cerad, mean_auc_cerad + 1.96 * se_auc_cerad)

        log_cerad_ors = np.log(cerad_odds_ratios)
        mean_log_or_cerad = np.mean(log_cerad_ors)
        se_log_or_cerad = np.std(log_cerad_ors, ddof=1) / np.sqrt(len(log_cerad_ors))
        or_cerad_ci = (
            np.exp(mean_log_or_cerad - 1.96 * se_log_or_cerad),
            np.exp(mean_log_or_cerad + 1.96 * se_log_or_cerad)
        )
        mean_or_cerad = np.exp(mean_log_or_cerad)

        _, combined_p_cerad = combine_pvalues(cerad_pvals, method='fisher')

        print(f"  CERAD mean AUC: {mean_auc_cerad:.3f} (95% CI: {auc_cerad_ci[0]:.3f}–{auc_cerad_ci[1]:.3f})")
        print(f"  CERAD mean OR: {mean_or_cerad:.2f} (95% CI: {or_cerad_ci[0]:.2f}–{or_cerad_ci[1]:.2f}), p={combined_p_cerad:.3g}")

        if mean_auc_cerad > 0.55:
            mean_fpr_cerad = np.linspace(0, 1, 100)
            interp_tprs_cerad = []

            for data in roc_plot_data_cerad:
                interp_tprs_cerad.append(np.interp(mean_fpr_cerad, data['fpr'], data['tpr']))

            mean_tpr_cerad = np.mean(interp_tprs_cerad, axis=0)
            mean_tpr_cerad[0], mean_tpr_cerad[-1] = 0, 1

            summary_roc_data_cerad.append({
                'fpr': mean_fpr_cerad,
                'tpr': mean_tpr_cerad,
                'Subcluster': current_subcluster,
                'mean_auc': mean_auc_cerad,
                'se_auc': se_auc_cerad
            })
    else:
        mean_auc_cerad = np.nan
        auc_cerad_ci = (np.nan, np.nan)
        mean_or_cerad = np.nan
        or_cerad_ci = (np.nan, np.nan)
        combined_p_cerad = np.nan

    # === Aggregate clinical AD stats across seeds ===
    mean_auc = np.mean(subcluster_aucs)
    se_auc = np.std(subcluster_aucs, ddof=1) / np.sqrt(len(subcluster_aucs))
    auc_ci_95 = (mean_auc - 1.96 * se_auc, mean_auc + 1.96 * se_auc)
    
    log_ors = np.log(subcluster_odds_ratios)
    mean_log_or = np.mean(log_ors)
    se_log_or = np.std(log_ors, ddof=1) / np.sqrt(len(log_ors))
    log_or_ci_95 = (mean_log_or - 1.96 * se_log_or, mean_log_or + 1.96 * se_log_or)
    mean_or = np.exp(mean_log_or)
    or_ci_95 = (np.exp(log_or_ci_95[0]), np.exp(log_or_ci_95[1]))

    _, combined_p_value = combine_pvalues(subcluster_pvals, method='fisher')
    
    print(f"\n--- Aggregated Results for {current_subcluster} ---")
    print(f"Mean AUC: {mean_auc:.4f} (95% CI: {auc_ci_95[0]:.4f} - {auc_ci_95[1]:.4f})")
    print(f"Mean Odds Ratio: {mean_or:.4f} (95% CI: {or_ci_95[0]:.4f} - {or_ci_95[1]:.4f})")
    print(f"Combined P-value (Fisher's): {combined_p_value:.4g}")

    # === Mean ROC curve across seeds for this subcluster ===
    if mean_auc > 0.55:
        mean_fpr = np.linspace(0, 1, 100)
        interp_tprs = []
        for fpr, tpr in zip(all_fprs, all_tprs):
            interp_tprs.append(np.interp(mean_fpr, fpr, tpr))
        
        if interp_tprs:
            mean_tpr = np.mean(interp_tprs, axis=0)
            mean_tpr[0], mean_tpr[-1] = 0, 1
            
            summary_roc_data.append({
                'fpr': mean_fpr,
                'tpr': mean_tpr,
                'Subcluster': current_subcluster,
                'mean_auc': mean_auc,
                'se_auc': se_auc
            })


    # Store per-subcluster summary row
    validation_results.append({
        'Subcluster': current_subcluster,
        'mean_auc': mean_auc,
        'auc_ci_lower': auc_ci_95[0],
        'auc_ci_upper': auc_ci_95[1],
        'mean_odds_ratio': mean_or,
        'or_ci_lower': or_ci_95[0],
        'or_ci_upper': or_ci_95[1],
        'combined_p_value': combined_p_value,
        'mean_auc_cerad': mean_auc_cerad,
        'auc_cerad_lower': auc_cerad_ci[0],
        'auc_cerad_upper': auc_cerad_ci[1],
        'odds_ratio_cerad': mean_or_cerad,
        'or_cerad_lower': or_cerad_ci[0],
        'or_cerad_upper': or_cerad_ci[1],
        'p_value_cerad': combined_p_cerad,
        'n_cells': adata_template.n_obs,
        'top_trajectory_genes': ', '.join(top_trajectory_genes_from_gt) if top_trajectory_genes_from_gt else "N/A",
        'top_pc_loadings': ', '.join(top_50_trajectory_genes_from_gt) if top_50_trajectory_genes_from_gt else "N/A",
    })

# --- Save Final Aggregated Validation Results ---
if not validation_results:
    print("\nNo subclusters were successfully processed. Exiting.")
else:
    validation_summary_df = pd.DataFrame(validation_results).sort_values(by='mean_auc', ascending=False)
    
    # FDR correction for clinical AD
    if 'combined_p_value' in validation_summary_df.columns:
        pvals = validation_summary_df['combined_p_value'].dropna()
        if not pvals.empty:
            reject, q_values, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')
            validation_summary_df['fdr_q_value'] = np.nan
            validation_summary_df.loc[pvals.index, 'fdr_q_value'] = q_values
    
    # FDR correction for CERAD
    if 'p_value_cerad' in validation_summary_df.columns:
        cerad_pvals = validation_summary_df['p_value_cerad'].dropna()
        if not cerad_pvals.empty:
            _, cerad_qvals, _, _ = multipletests(cerad_pvals, alpha=0.05, method='fdr_bh')
            validation_summary_df['fdr_q_value_cerad'] = np.nan
            validation_summary_df.loc[cerad_pvals.index, 'fdr_q_value_cerad'] = cerad_qvals

    validation_summary_path = os.path.join(output_path, "publication_summary_statistics.csv")
    validation_summary_df.to_csv(validation_summary_path, index=False)
    print(f"\nFinal publication-ready summary saved to: {validation_summary_path}")

    # === Final combined ROC plot for clinical AD (only subclusters with mean AUC > 0.55) ===
    if summary_roc_data:
        plt.style.use('seaborn-v0_8-whitegrid')
        fig, ax = plt.subplots(figsize=(10, 8))
        
        summary_roc_data.sort(key=lambda x: x['mean_auc'], reverse=True)
        
        for data in summary_roc_data:
            ax.plot(
                data['fpr'], data['tpr'],
                label=f"{data['Subcluster']} (AUC = {data['mean_auc']:.3f} \u00B1 {data['se_auc']:.3f})"
            )

        ax.plot([0, 1], [0, 1], 'k--', label='Chance (AUC = 0.50)')
        ax.set_xlabel('False Positive Rate', fontsize=14)
        ax.set_ylabel('True Positive Rate', fontsize=14)
        ax.set_title('Clinical AD ROC Curves using Disease Trajectory', fontsize=16)
        ax.legend(loc='lower right', frameon=True, fontsize=10)
        plt.savefig(os.path.join(output_path, "summary_roc_plot_all_subclusters.png"),
                    bbox_inches='tight', dpi=300)
        plt.close(fig)
        print(f"Saved summary ROC plot to: {os.path.join(output_path, 'summary_roc_plot_all_subclusters.png')}")

    # === Final combined ROC plot for CERAD (only subclusters with mean AUC > 0.55) ===
    if summary_roc_data_cerad:
        plt.style.use('seaborn-v0_8-whitegrid')
        fig, ax = plt.subplots(figsize=(10, 8))

        summary_roc_data_cerad.sort(key=lambda x: x['mean_auc'], reverse=True)

        for data in summary_roc_data_cerad:
            ax.plot(
                data['fpr'], data['tpr'],
                label=f"{data['Subcluster']} (AUC = {data['mean_auc']:.3f} ± {data['se_auc']:.3f})"
            )

        ax.plot([0, 1], [0, 1], 'k--', label='Chance (AUC = 0.50)')
        ax.set_xlabel('False Positive Rate', fontsize=14)
        ax.set_ylabel('True Positive Rate', fontsize=14)
        ax.set_title('Pathological AD ROC Curves using Disease Trajectory (CERAD)', fontsize=16)
        ax.legend(loc='lower right', frameon=True, fontsize=10)
        plt.savefig(os.path.join(output_path, "summary_roc_plot_CERAD_all_subclusters.png"),
                    bbox_inches='tight', dpi=300)
        plt.close(fig)
        print("Saved CERAD summary ROC plot to: summary_roc_plot_CERAD_all_subclusters.png")

print("\n--- Analysis complete. ---")
print("\n--- Pseudotime robustness analysis complete for all subclusters. ---")

END_TIME = time.time()
elapsed_hours = (END_TIME - START_TIME) / 3600

# ru_maxrss:
# - Linux (O2): KB
# - macOS: bytes
ru_maxrss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss

# Convert to GB safely (works on O2 + Mac)
if ru_maxrss > 1e9:   # macOS (bytes)
    peak_mem_gb = ru_maxrss / (1024**3)
else:                # Linux (KB)
    peak_mem_gb = ru_maxrss / (1024**2)

print("\n================ RESOURCE USAGE ================")
print(f"Total wall-clock runtime: {elapsed_hours:.2f} hours")
print(f"Peak resident memory:     {peak_mem_gb:.2f} GB")
print("================================================")

AD status defined based on 'dcfdx':
AD
Control    39202
AD         31432
Name: count, dtype: int64

--- Processing Subcluster: Ex8 ---
Selecting HVGs for pseudotime (non-circular approach)...
Using 2000 HVGs to build trajectories.
PCA complete: (1828, 30)
Building neighbor graph for 1828 cells...
Neighbor graph complete!


/tmp/ipykernel_2684789/2945074351.py:93: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata_template, resolution=0.5)



  -- Running analysis for seed: 41 --
  --> Seed 41: AUC = 0.5094
  CERAD AUC (seed 41): 0.514

  -- Running analysis for seed: 42 --
  --> Seed 42: AUC = 0.5116
  CERAD AUC (seed 42): 0.506

  -- Running analysis for seed: 43 --
  --> Seed 43: AUC = 0.4884
  CERAD AUC (seed 43): 0.506
  Top trajectory-informing genes (PC1): ['NEFL', 'NEFM', 'CLU', 'TUBB2A', 'HSP90AA1', 'FTH1', 'STMN1', 'UBB', 'PCSK1N', 'NDUFA4']
Saved top-50 PC loadings → /n/scratch/users/a/adm808/runtime_test_scraps/top50_PC_loadings_Ex8.csv
  CERAD mean AUC: 0.509 (95% CI: 0.504–0.513)
  CERAD mean OR: 0.93 (95% CI: 0.88–0.99), p=0.981

--- Aggregated Results for Ex8 ---
Mean AUC: 0.5031 (95% CI: 0.4886 - 0.5176)
Mean Odds Ratio: 0.8098 (95% CI: 0.6894 - 0.9511)
Combined P-value (Fisher's): 0.6679

--- Processing Subcluster: Ex0 ---
Selecting HVGs for pseudotime (non-circular approach)...
Using 2000 HVGs to build trajectories.
PCA complete: (7612, 30)
Building neighbor graph for 7612 cells...
Neighbor graph complet

/tmp/ipykernel_2684789/2945074351.py:320: RuntimeWarning: invalid value encountered in divide
  smoothed_z = (smoothed[:, 1] - smoothed[:, 1].mean()) / smoothed[:, 1].std()


  CERAD mean AUC: 0.498 (95% CI: 0.496–0.499)
  CERAD mean OR: 1.05 (95% CI: 1.04–1.06), p=0.998

--- Aggregated Results for In10 ---
Mean AUC: 0.5051 (95% CI: 0.5034 - 0.5067)
Mean Odds Ratio: 1.2371 (95% CI: 1.2290 - 1.2453)
Combined P-value (Fisher's): 0.8354

--- Processing Subcluster: In5 ---
Selecting HVGs for pseudotime (non-circular approach)...
Using 2000 HVGs to build trajectories.
PCA complete: (548, 30)
Building neighbor graph for 548 cells...
Neighbor graph complete!

  -- Running analysis for seed: 41 --
  --> Seed 41: AUC = 0.5792
  CERAD AUC (seed 41): 0.539

  -- Running analysis for seed: 42 --
  --> Seed 42: AUC = 0.5010
  CERAD AUC (seed 42): 0.468

  -- Running analysis for seed: 43 --
  --> Seed 43: AUC = 0.4894
  CERAD AUC (seed 43): 0.459
  Top trajectory-informing genes (PC2): ['NRG1', 'RELN', 'KCNIP4', 'C8orf34', 'TMEFF2', 'CDH13', 'ZBTB20', 'ASIC2', 'SORCS3', 'NEGR1']
Saved top-50 PC loadings → /n/scratch/users/a/adm808/runtime_test_scraps/top50_PC_loadings_I

In [12]:
import pandas as pd
import numpy as np

df = pd.read_csv("/home/adm808/pfc_deg_sacct_full.tsv", sep="|")

# Keep only COMPLETED steps
df = df[df["State"] == "COMPLETED"].copy()

# Keep batch steps only (this is where MaxRSS lives)
df_batch = df[df["JobID"].str.contains(r"\.batch|\.ba\+", regex=True)].copy()

# Extract parent JobID
df_batch["parent_jobid"] = df_batch["JobID"].str.replace(r"\.(batch|ba\+)", "", regex=True)

# ---- Parse elapsed ----
def elapsed_to_hours(s):
    if "-" in s:
        days, rest = s.split("-")
        h, m, sec = rest.split(":")
        return int(days)*24 + int(h) + int(m)/60 + int(sec)/3600
    else:
        h, m, sec = s.split(":")
        return int(h) + int(m)/60 + int(sec)/3600

df_batch["elapsed_hours"] = df_batch["Elapsed"].apply(elapsed_to_hours)

# MaxRSS already in GB because of --units=G
df_batch["maxrss_gb"] = pd.to_numeric(df_batch["MaxRSS"], errors="coerce")

# ---- Final summary ----
summary = {
    "n_jobs": df_batch["parent_jobid"].nunique(),
    "mean_hours": df_batch["elapsed_hours"].mean(),
    "median_hours": df_batch["elapsed_hours"].median(),
    "max_hours": df_batch["elapsed_hours"].max(),
    "mean_mem_gb": df_batch["maxrss_gb"].mean(),
    "median_mem_gb": df_batch["maxrss_gb"].median(),
    "max_mem_gb": df_batch["maxrss_gb"].max(),
}

summary_df = pd.DataFrame([summary]).round(3)

print("\n=== DEG JOB RESOURCE SUMMARY (FIXED) ===")
print(summary_df)

summary_df.to_csv("PFC_DEG_resource_summary.csv", index=False)


=== DEG JOB RESOURCE SUMMARY (FIXED) ===
   n_jobs  mean_hours  median_hours  max_hours  mean_mem_gb  median_mem_gb  \
0     162      35.465          24.8    182.702          0.0            0.0   

   max_mem_gb  
0         0.0  


In [13]:
import pandas as pd
import numpy as np

# ----------------------------
# Load sacct output
# ----------------------------
df = pd.read_csv(
    "/home/adm808/pfc_deg_sacct_full.tsv",
    sep="|",
    names=["JobID", "JobName", "Elapsed", "MaxRSS", "State"]
)

# ----------------------------
# Keep only COMPLETED batch steps
# ----------------------------
df = df[df["State"] == "COMPLETED"].copy()
df = df[df["JobID"].str.endswith(".batch")].copy()

# ----------------------------
# Extract parent job ID
# ----------------------------
df["parent_jobid"] = df["JobID"].str.replace(".batch", "", regex=False)

# ----------------------------
# Parse elapsed time → hours
# ----------------------------
def elapsed_to_hours(s):
    if "-" in s:
        days, rest = s.split("-")
        h, m, sec = rest.split(":")
        return int(days) * 24 + int(h) + int(m)/60 + int(sec)/3600
    else:
        h, m, sec = s.split(":")
        return int(h) + int(m)/60 + int(sec)/3600

df["elapsed_hours"] = df["Elapsed"].apply(elapsed_to_hours)

# ----------------------------
# Parse MaxRSS (already in GB)
# ----------------------------
df["maxrss_gb"] = (
    df["MaxRSS"]
    .astype(str)
    .str.replace("G", "", regex=False)
    .replace("", np.nan)
    .astype(float)
)

# Drop rows with missing or zero memory (safety)
df = df[df["maxrss_gb"] > 0]

# ----------------------------
# Final summary
# ----------------------------
summary = {
    "n_jobs": df["parent_jobid"].nunique(),
    "mean_hours": df["elapsed_hours"].mean(),
    "median_hours": df["elapsed_hours"].median(),
    "max_hours": df["elapsed_hours"].max(),
    "mean_mem_gb": df["maxrss_gb"].mean(),
    "median_mem_gb": df["maxrss_gb"].median(),
    "max_mem_gb": df["maxrss_gb"].max(),
}

summary_df = pd.DataFrame([summary]).round(3)

print("\n=== DEG JOB RESOURCE SUMMARY (CORRECT) ===")
print(summary_df)

summary_df.to_csv("PFC_DEG_resource_summary.csv", index=False)


=== DEG JOB RESOURCE SUMMARY (CORRECT) ===
   n_jobs  mean_hours  median_hours  max_hours  mean_mem_gb  median_mem_gb  \
0     161      35.685        24.869    182.702      105.669          77.24   

   max_mem_gb  
0      247.84  


In [14]:
import pandas as pd

# Paths (same as your analysis)
expr_path = "/n/groups/patel/adithya/CellMatrix_with_genenames.parquet"
meta_path = "/n/groups/patel/adithya/New_CellMetadataSyn1848517.parquet"

# Load metadata
meta_df = pd.read_parquet(meta_path)

# Define AD/control (same logic as analysis)
meta_df["AD"] = meta_df["dcfdx_lv"].isin([4.0, 5.0])

# Keep cohort used in modeling (48 donors)
donors = meta_df["projid"].dropna().unique()
print(f"Number of donors: {len(donors)}")

# Load expression
expr_df = pd.read_parquet(expr_path)
expr_df.set_index("index", inplace=True)   # genes x cells
expr_df = expr_df.T                        # cells x genes

# Subset to cells actually used
cells_used = meta_df["TAG"].values
expr_used = expr_df.loc[cells_used]

n_cells = expr_used.shape[0]
n_genes = expr_used.shape[1]

print("=== Predictive / Pseudotime / Coloc ===")
print(f"Cells used: {n_cells}")
print(f"Genes used (before HVG / feature selection): {n_genes}")
print(f"Matrix size: {n_genes} genes × {n_cells} cells")

Number of donors: 48
=== Predictive / Pseudotime / Coloc ===
Cells used: 70634
Genes used (before HVG / feature selection): 17926
Matrix size: 17926 genes × 70634 cells


In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

split_index = 1

CELL_TYPES = ["Ast", "Mic", "Opc", "Oli", "In", "Ex"]

# ----------------------------
# Load data once
# ----------------------------
metadata = pd.read_parquet('/home/adm808/New_CellMetadataSyn1848517.parquet')
metadata['alzheimers_or_control'] = metadata['dcfdx_lv'].isin([4.0, 5.0]).astype(int)

gene_matrix = pd.read_parquet('/home/adm808/NormalizedCellMatrixSyn18485175.parquet').T

# ----------------------------
# Train/test split (same for all cell types)
# ----------------------------
sample_summary = metadata[['sample', 'alzheimers_or_control', 'msex']].drop_duplicates()
sample_summary['stratify_group'] = (
    sample_summary['alzheimers_or_control'].astype(str) + "_" +
    sample_summary['msex'].astype(str)
)

train_samples, test_samples = train_test_split(
    sample_summary['sample'],
    test_size=0.2,
    random_state=split_index,
    stratify=sample_summary['stratify_group']
)

# ----------------------------
# Missing-gene filter
# ----------------------------
def select_missing_genes(filtered_matrix):
    mean_threshold = 2
    missingness_threshold = 90
    mean_gene_expression = filtered_matrix.mean(axis=0)
    missingness = (filtered_matrix == 0).sum(axis=0) / filtered_matrix.shape[0] * 100
    return filtered_matrix.columns[
        (missingness > missingness_threshold) &
        (mean_gene_expression < mean_threshold)
    ]

# ----------------------------
# Loop over cell types
# ----------------------------
rows = []

for cell_type in CELL_TYPES:
    train_metadata = metadata[
        (metadata['sample'].isin(train_samples)) &
        (metadata['broad.cell.type'] == cell_type)
    ]
    test_metadata = metadata[
        (metadata['sample'].isin(test_samples)) &
        (metadata['broad.cell.type'] == cell_type)
    ]

    train_matrix = gene_matrix.loc[train_metadata['TAG']]
    test_matrix  = gene_matrix.loc[test_metadata['TAG']]

    genes_to_drop = select_missing_genes(train_matrix)

    train_matrix = train_matrix.drop(columns=genes_to_drop)
    test_matrix  = test_matrix.drop(columns=[g for g in genes_to_drop if g in test_matrix.columns])

    rows.append({
        "cell_type": cell_type,
        "train_cells": train_matrix.shape[0],
        "test_cells": test_matrix.shape[0],
        "total_cells": train_matrix.shape[0] + test_matrix.shape[0],
        "genes_after_filtering": train_matrix.shape[1]
    })

summary_df = pd.DataFrame(rows)

print("\n=== FINAL MODEL INPUT SIZES (POST FILTERING) ===")
print(summary_df)

summary_df.to_csv("celltype_gene_cell_counts_preRFE.csv", index=False)


=== FINAL MODEL INPUT SIZES (POST FILTERING) ===
  cell_type  train_cells  test_cells  total_cells  genes_after_filtering
0       Ast         2685         707         3392                   2971
1       Mic         1454         466         1920                   1339
2       Opc         2064         563         2627                   3824
3       Oli        13587        4648        18235                   1751
4        In         7254        1942         9196                   5637
5        Ex        28484        6492        34976                   7591
